# 실습 3. 하이브리드 검색 — BM25(어휘) + 벡터(의미) + RRF

## 실습 목표

벡터(의미) 검색은 고유명사·코드·약어 같은 **표면형**에 약하고, BM25(키워드)는 동의어·의역에
약합니다(교안 4.1~4.3). 둘을 **RRF(Reciprocal Rank Fusion)** 로 합쳐 약점을 상호 보완합니다.

이번 실습은 **각 문서가 하나의 RAG 기법을 설명하는 코퍼스**(`RAG_TECH_DOCS`)를 써서,
**"정답 문서가 몇 위로 잡혔는가(rank)"** 로 BM25 vs Vector 의 강약점을 또렷이 비교합니다.

- 벡터 단독 / BM25 단독 / 하이브리드(RRF) 결과 비교
- **target(정답) 기반 blind spot** — 한쪽이 놓치는 문서를 다른 쪽이 잡는다

## 1. 준비 — RAG 기법 코퍼스 + 벡터·BM25 검색기

`RAG_TECH_DOCS` 는 doc01(벡터검색)·doc02(BM25)·doc04(리랭킹)…처럼 **id=정답**이 되도록
설계된 10개 문서입니다. `rank_of(target, docs)` 로 정답 문서의 순위를 잰다(낮을수록 좋음).

In [23]:
from _common import (rag_tech_documents, build_vector_retriever, build_bm25,
                     rrf_fuse, rank_of)

docs = rag_tech_documents()                 # 문서가 짧아 1청크=1문서(별도 청킹 불필요)
vec = build_vector_retriever(docs, k=10, name="d21_hybrid_nb")   # 의미(dense)
bm25 = build_bm25(docs, k=10)                                    # 어휘(sparse)

def ids(ds):    return [d.metadata.get("id", "?") for d in ds]
def topics(ds): return [d.metadata.get("topic", "?") for d in ds]

print(f"코퍼스 {len(docs)}문서 · 검색기 준비 완료")

코퍼스 10문서 · 검색기 준비 완료


## 2. 기본 비교 — 한 질의로 벡터/BM25/하이브리드 top-5

한 질의만 보면 두 방식이 비슷해 보일 수 있습니다. 다음 셀(3)에서 **정답 기반 시나리오**로
차이를 드러냅니다.

In [4]:
QUERY = "두 검색 방식을 합쳐 약점을 보완하는 방법"
v = vec.invoke(QUERY); b = bm25.invoke(QUERY)
fused = rrf_fuse([v, b], top_n=5)
print(f"① 벡터(의미) : {topics(v)[:5]}")
print(f"② BM25(어휘) : {topics(b)[:5]}")
print(f"③ 하이브리드 : {topics(fused)}")

① 벡터(의미) : ['하이브리드', '하이브리드', '하이브리드', '쿼리변환', '쿼리변환']
② BM25(어휘) : ['하이브리드', '청킹', 'BM25', '벡터검색', '쿼리변환']
③ 하이브리드 : ['하이브리드', '쿼리변환', 'BM25', '청킹', '벡터검색']


## 3. 정답(target) 기반 blind spot — 정답 문서의 순위 비교

각 시나리오는 한쪽 검색의 약점을 의도적으로 자극합니다. **정답 문서가 BM25/Vector/Hybrid에서
몇 위로 잡히는지**(낮을수록 좋음, ✗=top-10 밖) 봅니다.

In [5]:
SCENARIOS = [
    {"query": "키워드 검색의 한국어 처리", "target": "doc02",
     "expect": "BM25 우위 — 정답 doc02에 'BM25'가 직접 등장. Vector는 '한국어'에 끌려 다국어(doc05)로 샘"},
    {"query": "검색 결과를 좁은 후보에서 한 번 더 정렬하는 모델", "target": "doc04",
     "expect": "Vector 우위 — 정답 doc04의 단어가 질의와 거의 안 겹쳐 BM25는 놓침"},
]

def fmt(r): return f"{r:>3}" if r is not None else "  ✗"

for s in SCENARIOS:
    q, tgt = s["query"], s["target"]
    bm_r = rank_of(tgt, bm25.invoke(q))
    vc_r = rank_of(tgt, vec.invoke(q))
    hy_r = rank_of(tgt, rrf_fuse([vec.invoke(q), bm25.invoke(q)], top_n=10))
    print(f"🎯 query : {q!r}")
    print(f"   정답  : {tgt}  |  기대: {s['expect']}")
    print(f"   ── 정답 문서의 순위(낮을수록 좋음, ✗=top-10 밖) ──")
    print(f"     BM25 단독      : {fmt(bm_r)}")
    print(f"     Vector 단독    : {fmt(vc_r)}")
    print(f"     Hybrid(RRF)    : {fmt(hy_r)}\n")

🎯 query : '키워드 검색의 한국어 처리'
   정답  : doc02  |  기대: BM25 우위 — 정답 doc02에 'BM25'가 직접 등장. Vector는 '한국어'에 끌려 다국어(doc05)로 샘
   ── 정답 문서의 순위(낮을수록 좋음, ✗=top-10 밖) ──
     BM25 단독      :   1
     Vector 단독    :   ✗
     Hybrid(RRF)    :   5

🎯 query : '검색 결과를 좁은 후보에서 한 번 더 정렬하는 모델'
   정답  : doc04  |  기대: Vector 우위 — 정답 doc04의 단어가 질의와 거의 안 겹쳐 BM25는 놓침
   ── 정답 문서의 순위(낮을수록 좋음, ✗=top-10 밖) ──
     BM25 단독      :   9
     Vector 단독    :   1
     Hybrid(RRF)    :   1



**포인트 — Hybrid 의 강점은 '안전판'**

- **BM25 우위 시나리오**(doc02): Vector는 '한국어'에 끌려 정답을 5위로 밀지만, BM25는 'BM25' 정확
  키워드로 1위 → Hybrid가 2위로 회복
- **Vector 우위 시나리오**(doc04): BM25는 질의와 단어가 안 겹쳐 정답을 거의 놓치지만(10위), Vector는
  의미로 1위 → Hybrid가 살려냄
- 즉 Hybrid = **"두 방식 중 잘 잡은 쪽을 따라간다"** → 한쪽이 놓쳐도 다른 쪽이 보완하는 안전판
- (한국어 BM25는 공백 토큰화라 형태소 토크나이저 Kiwi/Mecab 권장)

# [TODO]

## [TODO] 1. BM25가 놓치는 질문을 Vector가 구제하는 사례

질의 **"답변 근거가 되는 부분만 추려 넣어 토큰을 아끼는 기법"**(정답 `doc07`·컨텍스트압축)은
정답 문서의 단어와 거의 안 겹쳐, **BM25는 정답을 top-1에서 놓칩니다.** BM25/Vector/Hybrid 의
정답 순위를 출력해 확인하세요. (위 셀의 `fmt`·`rank_of` 재사용)

In [8]:
q, tgt = "답변 근거가 되는 부분만 추려 넣어 토큰을 아끼는 기법", "doc07"
# [TODO] 1: 정답 doc07 의 BM25/Vector/Hybrid 순위 출력 (BM25는 top-1을 놓치는지 확인)
# [TODO] 여기에 코드를 작성하세요.
bm_r = rank_of(tgt, bm25.invoke(q))
vc_r = rank_of(tgt, vec.invoke(q))
hy_r = rank_of(tgt, rrf_fuse([vec.invoke(q), bm25.invoke(q)], top_n=10))

print(fmt(bm_r),fmt(vc_r),fmt(hy_r))

  4   1   1


## [TODO] 2. "정답을 가장 위로 올린 검색기" 판정 함수

세 검색기 중 **정답을 가장 위(작은 순위)로 올린 검색기 이름**을 돌려주는 `best_retriever(query,
target)` 를 작성하세요. 질의 **"모델을 속이려는 입력 공격을 막기"**(정답 `doc09`·보안)에 적용해
어느 검색기가 이 질문에 강한지 확인합니다. (미검색=None은 아주 큰 값으로 취급)

In [22]:
def best_retriever(query, target):
    ranks = {"BM25":   rank_of(target, bm25.invoke(query)),
             "Vector": rank_of(target, vec.invoke(query)),
             "Hybrid": rank_of(target, rrf_fuse([vec.invoke(query), bm25.invoke(query)], top_n=10))}
    # [TODO] 2: ranks 에서 순위가 가장 작은(최선) 검색기 이름과 ranks 딕셔너리를 반환
    # [TODO] 여기에 코드를 작성하세요.
    print(ranks)
    best = min(ranks, key=lambda name: ranks[name] if ranks[name] is not None else float("inf"))
    return best, ranks


name, ranks = best_retriever("모델을 속이려는 입력 공격을 막기", "doc09")
print("정답 doc09 순위:", ranks, "→ 최선:", name)   # Vector가 BM25보다 위 = 의미 검색이 강한 질문

{'BM25': 3, 'Vector': 1, 'Hybrid': 1}
정답 doc09 순위: {'BM25': 3, 'Vector': 1, 'Hybrid': 1} → 최선: Vector


## 실습 정리

- 벡터(의미)·BM25(표면형)의 **서로 다른 실패 지점**을 **정답 순위(rank)** 로 또렷이 확인
- Hybrid(RRF) = 한쪽이 놓쳐도 다른 쪽을 따라가는 **안전판**
- 한국어 BM25의 공백 토큰화 함정 → 형태소 토크나이저 권장